
# Caculation for RSSI 

Formule d=10^((ABS(RSSI)-A)/10*n)

- d=distance à l’émetteur 
- A=valeur absolue de la puissance à un mètre de distance à l ‘émetteur 
- n=coefficient de perte de trajet (n=2.2);
- RSSI=valeur de puissance reçu ;

 Sources : 
 https://projets-ima.plil.fr/mediawiki/index.php/Mesure_de_distance_par_RSSI#:~:text=D%27abord%2C%20on%20fait%20le,n

In [5]:
import math
import subprocess


def calcul_distance(RSSI, A, n):
    """
    Calcule la distance à partir du RSSI en utilisant la formule :
    d = 10^((ABS(RSSI) - A) / (10 * n))
    
    Args:
    - RSSI (float): Valeur du RSSI (Received Signal Strength Indication).
    - A (float): Valeur absolue de la puissance à un mètre de distance à l'émetteur.
    - n (float): Coefficient de perte de trajet.
    
    Returns:
    - float: La distance calculée en mètres.
    """
    distance = 10 ** ((abs(RSSI) - A) / (10 * n))
    return distance
#By Ip
def calcul_distance_with_ip(ip):
    """
    Calcule la distance à partir du RSSI en utilisant la formule :
    d = 10^((ABS(RSSI) - A) / (10 * n))
    
    Args:
    - ip (str): Adresse IP de l'émetteur.
    
    Returns:
    - float: La distance calculée en mètres.
    """
    # Ping (ms)
    ping_response = subprocess.Popen(["ping", "-c", "1", "-w", "1", ip], stdout=subprocess.PIPE).stdout.read()
    ping_response = ping_response.decode("latin-1")
    print("ping_response :", ping_response)
    # Extract ms 
    latency_ms = float(ping_response.split("time=")[1].split(" ms")[0])
    
    # Convert ms to m 
    speed_of_light = 299792458 # m/s (vitesse de la lumière)
    distance = (latency_ms / 1000) * (speed_of_light / 2)
    
    return distance


#Test POC 1 :
# extract_data_by_key(data, 'RSSI')
data = "Adresse MAC: adresseMac, RSSI: -10, Nom: NomTest, Adresse MAC: adresseMac2, RSSI: -102, Nom: NomTest2,   Adresse MAC: adresseMac3, RSSI: -103, Nom: NomTest3,"
# Rssi = extract_data_by_key(data,'RSSI')
# mac = extract_data_by_key(data,'MAC')
mac_ok =""
statut = False
#MAC adress ESP 
esp_macs= ['','','','','']
#Cherche si la MAc et égal a l'adresse d'un ESP
# for key, value in esp_macs:
#         if key == mac:
#             mac_ok = value
#             statut = True
#             break
#Si il y en a pas voir a combien de distance il ce trouve par rapport a l'ESP
# if statut == False:  
#     A = 50     
#     n = 2.2 
#     distance = calcul_distance(Rssi, A, n)

RSSI = -70  
A = 50     
n = 2.2    
# ip = "10.15.0.1"  

# ip = get_ip_from_mac("aa:6e:54:67:a3:39")
distance = calcul_distance(RSSI, A, n)
# distance_m_ip = calcul_distance_with_ip(ip)
# Ah tester
# distance_m_ip = calcul_distance_with_ip(mac_ok)

print("Distance correspondante (Téléphonne - ESP ) :", distance, "mètres")
# print("Distance en mètre par rapport a l'ip correspondante (ESP - Raspberry ) :", distance_m_ip, "mètres")
#Calcul des deux pour trouvé la distance entre le Rasberry au téléhphonne
# print("Distance (Téléphonne - Rasberry) : ", distance + distance_m_ip ," m" )


Distance correspondante (Téléphonne - ESP ) : 8.11130830789687 mètres


# Triangulation 

Prérequie : 

- Distance de chacun des ESP32 par rapport au Serveur(Raspberry)


In [ ]:
# Fonction pour effectuer la triangulation à partir des données RSSI de trois émetteurs
def triangulation(rssi1, rssi2, rssi3, A, n):
    # Coordonnées des émetteurs
    x1, y1 = 0, 0  # Coordonnées de l'émetteur 1
    x2, y2 = 3, 0  # Coordonnées de l'émetteur 2
    x3, y3 = 0, 4  # Coordonnées de l'émetteur 3
    
    # Calcul des distances à partir des signaux RSSI
    d1 = calcul_distance(rssi1, A, n)
    d2 = calcul_distance(rssi2, A, n)
    d3 = calcul_distance(rssi3, A, n)
    
    # Triangulation
    x = (d1**2 - d2**2 + x2**2) / (2*x2)
    y = (d1**2 - d3**2 + d3**2 - 2*x3*x + x3**2 + y3**2) / (2*y3)
    
    return x, y

# Exemple d'utilisation :
rssi1 = -70  
rssi2 = -75  
rssi3 = -80  
A = 50      
n = 2.2     

x, y = triangulation(rssi1, rssi2, rssi3, A, n)
print("Coordonnées estimées (x, y) :", (x, y))


   # Calcule la valeur absolue de la puissance à un mètre de distance (A)


In [ ]:
def calcul_A(d1, d2, d3, RSSI1, RSSI2, RSSI3, n):
    """
    Calcule la valeur absolue de la puissance à un mètre de distance (A)
    à partir des distances mesurées (d1, d2, d3) et des signaux RSSI (RSSI1, RSSI2, RSSI3).

    Args:
    - d1 (float): Distance mesurée à partir de l'émetteur 1 (en mètres).
    - d2 (float): Distance mesurée à partir de l'émetteur 2 (en mètres).
    - d3 (float): Distance mesurée à partir de l'émetteur 3 (en mètres).
    - RSSI1 (float): Valeur du RSSI (Received Signal Strength Indication) pour l'émetteur 1.
    - RSSI2 (float): Valeur du RSSI (Received Signal Strength Indication) pour l'émetteur 2.
    - RSSI3 (float): Valeur du RSSI (Received Signal Strength Indication) pour l'émetteur 3.
    - n (float): Coefficient de perte de trajet.

    Returns:
    - float: La valeur absolue de la puissance à un mètre de distance (A).
    """
    A = (abs(RSSI1) - 10 * n * math.log10(d1) + abs(RSSI2) - 10 * n * math.log10(d2) + abs(RSSI3) - 10 * n * math.log10(d3)) / 3
    return A

# Exemple d'utilisation :
d1 = 1.5  
d2 = 2.5  
d3 = 3.5  
RSSI1 = -70  
RSSI2 = -75  
RSSI3 = -80  
n = 2.2    

A = calcul_A(d1, d2, d3, RSSI1, RSSI2, RSSI3, n)
print("Valeur absolue de la puissance à un mètre de distance (A) :", A)


# Regroup all function 

In [ ]:
def triangulation(rssi1, rssi2, rssi3, d1, d2, d3, n):
    # Calcul de la valeur absolue de la puissance à un mètre de distance (A)
    A = calcul_A(d1, d2, d3, rssi1, rssi2, rssi3, n)
    
    # Coordonnées des émetteurs
    x1, y1 = 0, 0  
    x2, y2 = 3, 0  
    x3, y3 = 0, 4 
    
    # Calcul des distances à partir des signaux RSSI
    d1 = calcul_distance(rssi1, A, n)
    d2 = calcul_distance(rssi2, A, n)
    d3 = calcul_distance(rssi3, A, n)
    
    # Triangulation
    x = (d1**2 - d2**2 + x2**2) / (2*x2)
    y = (d1**2 - d3**2 + d3**2 - 2*x3*x + x3**2 + y3**2) / (2*y3)
    
    return x, y

# data revoyer par ESP 
# data = 

# RSSI a calculé avec la fonction a partir de l'adresse mac de ESP32 
rssi1 = -75  
rssi2 = -105  
rssi3 = -50 
# rssi1 = extract_data_by_key(data, 'RSSI')
# rssi2 = extract_data_by_key(data, 'RSSI') 
# rssi3 = extract_data_by_key(data, 'RSSI') 

#Distance fix 
d1 = 1.5  
d2 = 2.5  
d3 = 3.5 
# Distance en fonction l'ip
# d1 = calcul_distance_with_ip(extract_data_by_key(data, 'MAC'))
# d2 = calcul_distance_with_ip(extract_data_by_key(data, 'MAC')) 
# d3 = calcul_distance_with_ip(extract_data_by_key(data, 'MAC')) 
#Valeur fix 
n = 2.2   

x, y = triangulation(rssi1, rssi2, rssi3, d1, d2, d3, n)
print("Coordonnées estimées (x, y) :", (x, y))


# Recept data with ESP32

In [ ]:
import re
#All
def extract_data(string):
    """
    Extrait les données entre les caractères ":" et "," dans la chaîne de caractères donnée.
    
    Args:
    - string (str): Chaîne de caractères contenant les données.
    
    Returns:
    - list: Liste des données extraites.
    """
    matches = re.findall(r"(\w+):\s*([^,]+),", string)
    if matches:
        return matches
    else:
        return None
# 
def extract_data_by_key(string, key):
    """
    Extrait toutes les valeurs associées à la clé donnée dans la chaîne de caractères donnée.
    
    Args:
    - string (str): Chaîne de caractères contenant les données.
    - key (str): Clé dont vous voulez récupérer les valeurs associées.
    
    Returns:
    - list: Liste des valeurs associées à la clé donnée.
    """
    matches = re.findall(r"%s:\s*([^,]+)," % key, string)
    return matches


#Test
data = "Adresse MAC: adresseMac, RSSI: -10, Nom: NomTest, Adresse MAC: adresseMac2, RSSI: -102, Nom: NomTest2,   Adresse MAC: adresseMac3, RSSI: -103, Nom: NomTest3,"

data_extracted = extract_data(data)
test = extract_data_by_key(data , 'RSSI')
print('test : ',test)
if data_extracted is not None:
    print("Données extraites :", extract_data(data))
else:
    print("Aucune donnée trouvée dans la chaîne.")


# Adress Mac to Ip

In [ ]:
def get_ip_from_mac(mac_address):
    """
    Récupère l'adresse IP associée à une adresse MAC dans la table ARP.

    Args:
    - mac_address (str): Adresse MAC.

    Returns:
    - str: Adresse IP associée à l'adresse MAC, ou None si aucune adresse IP n'est trouvée.
    """
    # Exécute la commande 'arp -a' pour obtenir la table ARP
    arp_table = subprocess.Popen(["arp", "-a"], stdout=subprocess.PIPE).stdout.read().decode("utf-8")

    # Recherche de l'adresse IP dans la table ARP
    match = re.search(r"(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s+([0-9a-fA-F]{2}[:-]){5}([0-9a-fA-F]{2})", arp_table)
    while match:
        ip_address, found_mac = match.group(1), match.group(0)
        if found_mac.lower() == mac_address.lower():
            return ip_address
        arp_table = arp_table[match.end():]
        match = re.search(r"(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\s+([0-9a-fA-F]{2}[:-]){5}([0-9a-fA-F]{2})", arp_table)

    return None


In [1]:
import subprocess
import nmap



def calcul_distance(RSSI, A, n):
    """
    Calcule la distance à partir du RSSI en utilisant la formule :
    d = 10^((ABS(RSSI) - A) / (10 * n))
    
    Args:
    - RSSI (float): Valeur du RSSI (Received Signal Strength Indication).
    - A (float): Valeur absolue de la puissance à un mètre de distance à l'émetteur.
    - n (float): Coefficient de perte de trajet.
    
    Returns:
    - float: La distance calculée en mètres.
    """
    distance = 10 ** ((abs(RSSI) - A) / (10 * n))
    return distance

#By Ip
def calcul_distance_with_ip(ip):
    """
    Calcule la distance à partir du RSSI en utilisant la formule :
    d = 10^((ABS(RSSI) - A) / (10 * n))
    
    Args:
    - ip (str): Adresse IP de l'émetteur.
    
    Returns:
    - float: La distance calculée en mètres.
    """
    # Ping (ms)
    ping_response = subprocess.Popen(["ping", "-c", "1", "-w", "1", ip], stdout=subprocess.PIPE).stdout.read()
    ping_response = ping_response.decode("latin-1")
    print("ping_response :", ping_response)
    # Extract ms 
    latency_ms = float(ping_response.split("time=")[1].split(" ms")[0])
    
    # Convert ms to m 
    speed_of_light = 299792458 # m/s (vitesse de la lumière)
    distance = (latency_ms / 1000) * (speed_of_light / 2)
    
    return distance


def get_ip_from_mac(mac_address):
    # Créez un objet `nmap.PortScanner()`
    nm = nmap.PortScanner()

    # Scannez le réseau local pour obtenir les adresses IP associées aux adresses MAC
    nm.scan(hosts="192.168.1.0/24", arguments="-sP")

    # Recherchez l'adresse IP associée à l'adresse MAC spécifiée
    for _, host in nm.all_hosts().items():
        if 'addresses' in host and 'mac' in host['addresses']:
            if host['addresses']['mac'].lower() == mac_address.lower():
                return host['addresses']['ipv4']

    return None

#Test POC 1 :
# extract_data_by_key(data, 'RSSI')
data = "Adresse MAC: adresseMac, RSSI: -10, Nom: NomTest, Adresse MAC: adresseMac2, RSSI: -102, Nom: NomTest2,   Adresse MAC: adresseMac3, RSSI: -103, Nom: NomTest3,"
Rssi = extract_data_by_key(data,'RSSI')
mac = extract_data_by_key(data,'MAC')
mac_ok =""
statut = False
#MAC adress ESP 
esp_macs= ['','','','','']
#Cherche si la MAc et égal a l'adresse d'un ESP
# for key, value in esp_macs:
#         if key == mac:
#             mac_ok = value
#             statut = True
#             break
#Si il y en a pas voir a combien de distance il ce trouve par rapport a l'ESP
# if statut == False:  
#     A = 50     
#     n = 2.2 
#     distance = calcul_distance(Rssi, A, n)

RSSI = -70  
A = 50     
n = 2.2    
# ip = "10.15.0.1"  

ip = get_ip_from_mac("aa:6e:54:67:a3:39")
if ip:
    distance = calcul_distance(RSSI, A, n)
    distance_m_ip = calcul_distance_with_ip(ip)
    # Ah tester
    # distance_m_ip = calcul_distance_with_ip(mac_ok)

    print("Distance correspondante (Téléphonne - ESP ) :", distance, "mètres")
    print("Distance en mètre par rapport a l'ip correspondante (ESP - Raspberry ) :", distance_m_ip, "mètres")
    #Calcul des deux pour trouvé la distance entre le Rasberry au téléhphonne
    print("Distance (Téléphonne - Rasberry) : ", distance + distance_m_ip ," m" )
else:
    print("Aucune adresse IP associée à cette adresse MAC.")


NameError: name 'extract_data_by_key' is not defined